# VGG16 — Feature Extraction & Invariance Analysis (raw / no pooling)

Same as `invariance_VGG16.ipynb` but features are **not** globally average-pooled.
Each activation map is flattened to `(B, C×H×W)` before computing the representation similarity matrix.

Two granularities:

| Granularity | # levels | Description |
|-------------|-----------|-------------|
| **Per-block** | 5 | last conv of each block = after MaxPool at block boundary |
| **Per-conv** | 13 | input to next conv: after ReLU for mid-block convs, after MaxPool for last conv in each block |

In [1]:
import os, sys
import torch
import numpy as np
from PIL import Image
import torchvision.models as models
import torchvision.transforms as T
from collections import OrderedDict
import matplotlib.pyplot as plt

sys.path.append('../utils')
import util

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cuda


## Load VGG16

In [2]:
weights = models.VGG16_Weights.IMAGENET1K_V1
model = models.vgg16(weights=weights).to(device)
model.eval()

conv_defs = [
    ('B1.C1',  1),   # ReLU   — input to Conv(2)
    ('B1.C2',  4),   # MaxPool — input to Conv(5)
    ('B2.C1',  6),   # ReLU   — input to Conv(7)
    ('B2.C2',  9),   # MaxPool — input to Conv(10)
    ('B3.C1', 11),   # ReLU   — input to Conv(12)
    ('B3.C2', 13),   # ReLU   — input to Conv(14)
    ('B3.C3', 16),   # MaxPool — input to Conv(17)
    ('B4.C1', 18),   # ReLU   — input to Conv(19)
    ('B4.C2', 20),   # ReLU   — input to Conv(21)
    ('B4.C3', 23),   # MaxPool — input to Conv(24)
    ('B5.C1', 25),   # ReLU   — input to Conv(26)
    ('B5.C2', 27),   # ReLU   — input to Conv(28)
    ('B5.C3', 30),   # MaxPool — input to classifier
]

block_last_conv   = ['B1.C2', 'B2.C2', 'B3.C3', 'B4.C3', 'B5.C3']
n_convs_per_block = [2, 2, 3, 3, 3]

print('VGG16 per-conv hook positions:')
for cname, idx in conv_defs:
    print(f'  {cname:8s}: features[{idx:2d}] = {model.features[idx].__class__.__name__}')
print(f'\nTotal conv layers : {len(conv_defs)}')

VGG16 per-conv hook positions:
  B1.C1   : features[ 1] = ReLU
  B1.C2   : features[ 4] = MaxPool2d
  B2.C1   : features[ 6] = ReLU
  B2.C2   : features[ 9] = MaxPool2d
  B3.C1   : features[11] = ReLU
  B3.C2   : features[13] = ReLU
  B3.C3   : features[16] = MaxPool2d
  B4.C1   : features[18] = ReLU
  B4.C2   : features[20] = ReLU
  B4.C3   : features[23] = MaxPool2d
  B5.C1   : features[25] = ReLU
  B5.C2   : features[27] = ReLU
  B5.C3   : features[30] = MaxPool2d

Total conv layers : 13


## Load & preprocess images

In [3]:
from scipy.io import loadmat

datapath = '../data'
mat = loadmat(os.path.join(datapath, 'miguel_passive8x4.mat'))
img = mat['img'].astype(np.float32)
images_gray = np.transpose(img, (2, 0, 1))   # (32, 150, 600)
print('images_gray:', images_gray.shape, '  range:', images_gray.min(), images_gray.max())

nimg = len(images_gray)
print(f'nimg={nimg}  (8 categories x 4 instances)')

images_rgb = [Image.fromarray(images_gray[i]).convert('RGB') for i in range(nimg)]

images_gray: (32, 150, 600)   range: 0.0 255.0
nimg=32  (8 categories x 4 instances)


In [4]:
mean_norm = (0.485, 0.456, 0.406)
std_norm  = (0.229, 0.224, 0.225)

transform = T.Compose([
    T.Resize((64, 264)),
    T.ToTensor(),
    T.Normalize(mean=mean_norm, std=std_norm),
])

x = torch.stack([transform(im) for im in images_rgb], dim=0).to(device)
print('Input batch:', x.shape, x.dtype)

Input batch: torch.Size([32, 3, 64, 264]) torch.float32


## Register hooks and extract features

In [5]:
feats_per_conv = OrderedDict()
handles = []

def make_hook(store, name):
    def _h(m, inp, out):
        store[name] = out.detach().cpu().clone()
    return _h

for cname, idx in conv_defs:
    handles.append(model.features[idx].register_forward_hook(make_hook(feats_per_conv, cname)))

with torch.no_grad():
    _ = model(x)

for h in handles:
    h.remove()

# Per-block: last conv of each block is already in per-conv
feats_per_block = OrderedDict()
for bname, lc in zip(['B1', 'B2', 'B3', 'B4', 'B5'], block_last_conv):
    feats_per_block[bname] = feats_per_conv[lc]

print(f'Per-conv  : {len(feats_per_conv)} features')
print(f'Per-block : {len(feats_per_block)} features (= last conv of each block)')
print('\nRaw activation map shapes:')
for k, v in feats_per_conv.items():
    print(f'  {k:8s}: {tuple(v.shape)}')
print('  ---')
for k, v in feats_per_block.items():
    print(f'  {k:8s}: {tuple(v.shape)}')

Per-conv  : 13 features
Per-block : 5 features (= last conv of each block)

Raw activation map shapes:
  B1.C1   : (32, 64, 64, 264)
  B1.C2   : (32, 64, 32, 132)
  B2.C1   : (32, 128, 32, 132)
  B2.C2   : (32, 128, 16, 66)
  B3.C1   : (32, 256, 16, 66)
  B3.C2   : (32, 256, 16, 66)
  B3.C3   : (32, 256, 8, 33)
  B4.C1   : (32, 512, 8, 33)
  B4.C2   : (32, 512, 8, 33)
  B4.C3   : (32, 512, 4, 16)
  B5.C1   : (32, 512, 4, 16)
  B5.C2   : (32, 512, 4, 16)
  B5.C3   : (32, 512, 2, 8)
  ---
  B1      : (32, 64, 32, 132)
  B2      : (32, 128, 16, 66)
  B3      : (32, 256, 8, 33)
  B4      : (32, 512, 4, 16)
  B5      : (32, 512, 2, 8)


## Flatten spatial dims → `(B, C×H×W)` feature vectors

No pooling — the full spatial layout is preserved in the flattened vector.

In [6]:
def flat_features(feats_dict):
    """(B, C, H, W) -> (B, C*H*W) by flattening all non-batch dims."""
    flat = OrderedDict()
    for name, t in feats_dict.items():
        flat[name] = t.reshape(t.shape[0], -1).numpy()
    return flat

flat_conv  = flat_features(feats_per_conv)
flat_block = flat_features(feats_per_block)

print('Per-block flat shapes (B, C*H*W):')
for k, v in flat_block.items():
    print(f'  {k}: {v.shape}')

Per-block flat shapes (B, C*H*W):
  B1: (32, 270336)
  B2: (32, 135168)
  B3: (32, 67584)
  B4: (32, 32768)
  B5: (32, 8192)


## Compute invariance

In [7]:
def compute_invariance(flat_dict):
    keys = list(flat_dict.keys())
    arr = np.empty(len(keys), dtype=object)
    for i, k in enumerate(keys):
        arr[i] = flat_dict[k]   # (32, D)
    rep_mtx  = util.compute_model_rep_mtx(arr)
    invar_df = util.compute_pair_inv_model(rep_mtx)
    return invar_df

print('Computing per-block invariance...')
invar_df_block = compute_invariance(flat_block)
print('Computing per-conv invariance ...')
invar_df_conv  = compute_invariance(flat_conv)
print('Done.')

Computing per-block invariance...
Computing per-conv invariance ...
Done.


## Summarise (mean ± std per layer) and save

In [8]:
def summarise_inv(invar_df, layer_names):
    grp = invar_df.groupby('layer')['pair_invariance']
    return {
        'layer_names': layer_names,
        'mean': grp.mean().values,
        'std':  grp.std().values,
    }

keys_block = list(flat_block.keys())
keys_conv  = list(flat_conv.keys())

inv_block = summarise_inv(invar_df_block, keys_block)
inv_conv  = summarise_inv(invar_df_conv,  keys_conv)

os.makedirs('outputs', exist_ok=True)
np.save('outputs/vgg16_inv_per_block_raw.npy', inv_block, allow_pickle=True)
np.save('outputs/vgg16_inv_per_conv_raw.npy',  inv_conv,  allow_pickle=True)
print('Invariance summaries saved.')
print('\nPer-block mean invariance:')
for name, m, s in zip(keys_block, inv_block['mean'], inv_block['std']):
    print(f'  {name}: {m:.4f} ± {s:.4f}')

Invariance summaries saved.

Per-block mean invariance:
  B1: 0.0021 ± 0.0050
  B2: 0.0323 ± 0.0309
  B3: 0.1201 ± 0.0783
  B4: 0.2785 ± 0.0852
  B5: 0.4012 ± 0.0919


## Save flat features

In [9]:
def save_flat_arr(flat_dict, fname):
    arr = np.empty(len(flat_dict), dtype=object)
    for i, v in enumerate(flat_dict.values()):
        arr[i] = v
    np.save(fname, arr, allow_pickle=True)
    print('saved:', fname)

save_flat_arr(flat_block, 'outputs/vgg16_flat_features_block.npy')
save_flat_arr(flat_conv,  'outputs/vgg16_flat_features_conv.npy')

saved: outputs/vgg16_flat_features_block.npy
saved: outputs/vgg16_flat_features_conv.npy


## Plot invariance across both granularities

In [ ]:
def plot_inv_summary(ax, inv, title, color='#e0a917', label_rotation=45, label_fs=9):
    names = inv['layer_names']
    mean  = inv['mean']
    std   = inv['std']
    x     = np.arange(len(names))
    ax.plot(x, mean, 'o-', color=color, lw=2, ms=5)
    ax.fill_between(x, mean - std, mean + std, alpha=0.2, color=color)
    ax.axhline(0, color='gray', lw=1, linestyle='--')
    ax.set_ylim(-0.1, 0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=label_rotation, ha='right', fontsize=label_fs)
    ax.set_title(title, fontsize=12)
    ax.set_ylabel('Invariance')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


def add_block_boundaries(ax, n_per_block):
    offset = 0
    for bname, nc in zip(['B1', 'B2', 'B3', 'B4', 'B5'], n_per_block):
        ax.axvline(offset - 0.5, color='gray', lw=0.8, linestyle=':', alpha=0.7)
        ax.text(offset + 0.1, 0.97, bname,
                transform=ax.get_xaxis_transform(),
                ha='left', va='top', fontsize=9, color='gray')
        offset += nc


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_inv_summary(axes[0], inv_block,
                 f'Per-Block  ({len(keys_block)} levels)',
                 color='#e0a917', label_rotation=30, label_fs=11)

plot_inv_summary(axes[1], inv_conv,
                 f'Per-Conv  ({len(keys_conv)} levels)',
                 color='#e0a917', label_rotation=45, label_fs=8)
add_block_boundaries(axes[1], n_convs_per_block)

plt.suptitle('VGG16 — Invariance Across Layers', fontsize=14)
plt.tight_layout()
plt.savefig('../figures/vgg16_invariance_raw.png', dpi=300, bbox_inches='tight')
plt.show()